In [1]:
import ee
import geemap
import pandas as pd
import os
import json
import geopandas as gpd
import time

# Initialize your project name to run this code
ee.Authenticate()
ee.Initialize(project='unicef-ccri')

In [ ]:
# =========================================================
# 1. SETUP ASSETS AND BOUNDARIES
# =========================================================

# Child Population
org_childpop = ee.ImageCollection("projects/unicef-ccri/assets/population/worldpop_T_U18_2025_CN_100m")
childpop = org_childpop.mosaic().setDefaultProjection(
    org_childpop.first().projection()
)
pop_res = org_childpop.first().projection().nominalScale()

# General Population
org_pop = ee.ImageCollection("projects/unicef-ccri/assets/population/worldpop_T_2025_CN_100m")
pop = org_pop.mosaic().setDefaultProjection(
    org_pop.first().projection()
)

# Load Boundaries
detailedBoundaries = ee.FeatureCollection('projects/unicef-ccri/assets/global_boundary/adm0_chunked_500km_shp')

In [3]:
# =========================================================
# 2. HAZARD DEFINITIONS
# =========================================================

hazards = [
    {"id": "projects/unicef-ccri/assets/hazards/river_flood_r100", "threshold": 0.01, "name": "river_flood_100yr_jrc_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/coastal_flood_r100", "threshold": 0, "name": "coastal_flood_100yr_jrc_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/storm_giri_rp100", "threshold": 17.5, "name": "tropical_storm_100yr_giri_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/ASI_return_level_100yr", "threshold": 30, "name": "agricultural_drought_fao_1984-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/droughts/spei12_TerraClimate_1958-2025", "band": "b2", "threshold": 0.06501621521265394, "name": "drought_spei_terraclimate_1958-2025", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/droughts/spi12_TerraClimate_1958-2025", "band": "b2", "threshold": 0.09128389509009999, "name": "drought_spi_terraclimate_1958-2025", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_frequency_return_level_100yr", "threshold": 16.02581287445288, "name": "heatwave_frequency_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_duration_return_level_100yr", "threshold": 94.01266417971051, "name": "heatwave_duration_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_severity_return_level_100yr", "threshold": 3.659148767122408, "name": "heatwave_severity_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/high_temp_degree_days_return_level_100yr", "threshold": 35, "name": "extreme_heat_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/FIRMS_FRP_90th_percentile", "threshold": 37.89412565331788, "name": "fire_FRP_nasa_2001-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/FIRMS_count_90th_percentile", "threshold": 4.9134115164042935, "name": "fire_frequency_nasa_2001-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/sand_dust_storm_annual", "threshold": 0, "name": "sand_dust_storm_unccd_2024", "type": "Image", "no_data": -999000000},
    {"id": "projects/unicef-ccri/assets/hazards/pm25_p90_1998_2023", "threshold": 5, "name": "air_pollution_pm25_1998-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/Pv_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapv_2012-2022", "type": "Image", "no_data": -9999},
    {"id": "projects/unicef-ccri/assets/hazards/Pf_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapf_2012-2022", "type": "Image", "no_data": -9999},
    {"id": "projects/unicef-ccri/assets/JBA_FLSW", "threshold": 0.01, "name": "pluvial_flood", "type": "Collection"}
]

In [4]:
# =========================================================
# 3. MHI SETUP — COMPUTE GLOBAL PERCENTILE THRESHOLDS
# =========================================================

mhi_image = ee.Image('projects/unicef-ccri/assets/hazards/MHI_climate')
mhi_percentiles = [75, 80, 85, 90, 95]

# Resolve band name (single-band GEE asset is typically 'b1')
mhi_band = mhi_image.bandNames().getInfo()[0]
print(f"MHI band name: {mhi_band}")

# Compute global percentile thresholds on land pixels (ocean is masked)
mhi_percentile_vals = mhi_image.reduceRegion(
    reducer=ee.Reducer.percentile(mhi_percentiles),
    geometry=ee.Geometry.BBox(-180, -90, 180, 90),
    scale=mhi_image.projection().nominalScale(),
    maxPixels=1e13,
    bestEffort=True
).getInfo()

print("MHI global percentile thresholds:")
for k, v in mhi_percentile_vals.items():
    print(f"  {k}: {v:.4f}")

MHI band name: b1
MHI global percentile thresholds:
  b1_p75: 5.2188
  b1_p80: 5.5937
  b1_p85: 5.9060
  b1_p90: 6.3436
  b1_p95: 6.9057


In [5]:
# =========================================================
# 3. BINARY MASK GENERATOR
# =========================================================

def get_binary_mask(h):
    # 1. Load the layer
    layer = (
        ee.ImageCollection(h['id']).mosaic()
        if h['type'] == 'Collection'
        else ee.Image(h['id'])
    )
    # 2. Select band if specified
    if 'band' in h:
        layer = layer.select(h['band'])

    th = h['threshold']
    # 3. HANDLE EXPLICIT NODATA (Crucial for Malaria -9999)
    # If the dictionary has 'no_data', mask those pixels immediately.
    if 'no_data' in h:
        layer = layer.updateMask(layer.neq(h['no_data']))

    # FAO drought index: Clean 0-100 range first
    if h['name'] == "agri_drought_fao":
        # Mask out invalid values (e.g. > 100) so they don't count as "Safe"
        layer = layer.updateMask(layer.gte(0).And(layer.lte(100)))
        return layer.gt(th)

    # SPI / SPEI (TerraClimate): values >= mean indicate drought exposure
    if "spei" in h['name'] or "spi" in h['name']:
        return layer.gte(th)

    # Default hazards
    binary = layer.gt(th)

    # Ensure the binary result inherits the mask of the input layer
    return binary.updateMask(layer.mask())

In [ ]:
# =========================================================
# 4. GENERATE ANALYSIS LAYERS
# =========================================================

hazardTopics = {
    'river_flood': ['river_flood_100yr_jrc_2024'],
    'coastal_flood': ['coastal_flood_100yr_jrc_2024'],
    'pluvial_flood': ['pluvial_flood'],
    'storm': ['tropical_storm_100yr_giri_2024'],
    'drought': ['agricultural_drought_fao_1984-2023', 'drought_spei_terraclimate_1958-2025', 'drought_spi_terraclimate_1958-2025'],
    'heatwave': ['heatwave_frequency_ecmwf_2014-2024', 'heatwave_duration_ecmwf_2014-2024', 'heatwave_severity_ecmwf_2014-2024'],
    'extreme_heat': ['extreme_heat_ecmwf_2014-2024'],
    'fire': ['fire_FRP_nasa_2001-2024', 'fire_frequency_nasa_2001-2023'],
    'sand_dust': ['sand_dust_storm_unccd_2024'],
    'air_pollution': ['air_pollution_pm25_1998-2023'],
    'malaria': ['vectorborne_malariapv_2012-2022', 'vectorborne_malariapf_2012-2022']
}

topic_exposure_list = ['drought', 'heatwave', 'fire']

exposureByHazardName = {h['name']: get_binary_mask(h) for h in hazards}

# Topic union masks
topicExposureImages = {}
for topic, names in hazardTopics.items():
    masks = [exposureByHazardName[name].unmask(0) for name in names]
    unionMask = masks[0]
    for m in masks[1:]:
        unionMask = unionMask.Or(m)
    topicExposureImages[topic] = unionMask

# Topic count per pixel
topicCount = ee.Image.cat(list(topicExposureImages.values())).reduce(ee.Reducer.sum())

# For each hazard: exposed population band + valid pixel band
popExposed = {}
validMasks = {}
for h in hazards:
    name = h['name']
    mask = exposureByHazardName[name]
    popExposed[name] = childpop.multiply(mask).unmask(0).rename(f'pop_exposed_{name}')
    validMasks[name] = mask.mask().unmask(0).rename(f'valid_{name}')

# Global valid: sum of all valid masks — > 0 means at least one hazard has data
globalValid = ee.Image.cat(list(validMasks.values())).reduce(ee.Reducer.sum()).rename('valid_global')

# Per-topic valid bands: OR of valid masks for each topic's hazards
topicValidBands = []
for topic in topic_exposure_list:
    names = hazardTopics[topic]
    valid_union = validMasks[names[0]]
    for n in names[1:]:
        valid_union = valid_union.Or(validMasks[n])
    topicValidBands.append(valid_union.rename(f'valid_{topic}_topic'))

# Topic count bands: children exposed to >= n topics
topicCountBands = [
    childpop.multiply(topicCount.gte(n)).unmask(0).rename(f'pop_topic_ge_{n}')
    for n in range(1, 9)
]

# Topic exposure bands (drought, heatwave, fire)
topicExposureBands = [
    childpop.multiply(topicExposureImages[topic]).unmask(0).rename(f'pop_exposed_{topic}_topic')
    for topic in topic_exposure_list
]

# MHI exposure bands — one per percentile threshold
mhi_valid = mhi_image.mask().unmask(0).rename('valid_mhi')
mhi_exposureBands = []
for p in mhi_percentiles:
    threshold = mhi_percentile_vals[f'{mhi_band}_p{p}']
    exposed = mhi_image.gt(threshold)
    mhi_exposureBands.append(childpop.multiply(exposed).unmask(0).rename(f'pop_mhi_p{p}'))

# Combine into multi-band image
bands = [childpop.rename('pop_child_total'), pop.rename('pop_total'), globalValid]
for name in exposureByHazardName.keys():
    bands.append(popExposed[name])
    bands.append(validMasks[name])
bands.extend(topicCountBands)
bands.extend(topicExposureBands)
bands.extend(topicValidBands)
bands.append(mhi_valid)
bands.extend(mhi_exposureBands)

analysisImage = ee.Image(bands)

In [ ]:
# =========================================================
# 5. ZONAL STATISTICS BY COUNTRY
# =========================================================

countryStats = analysisImage.reduceRegions(
  collection=detailedBoundaries,
  reducer=ee.Reducer.sum(),
  scale=pop_res,
  tileScale=1
)

def process_feature(feature):
    def safeGet(propName):
        value = feature.get(propName)
        return ee.Number(ee.Algorithms.If(ee.Algorithms.IsEqual(value, None), 0, value))

    results = {
        'ucode': feature.get('ucode'),
        'pop_child_total': safeGet('pop_child_total'),
        'pop_total': safeGet('pop_total')
    }

    # Individual hazard exposures: None if no valid pixels in country
    for h in hazards:
        name = h['name']
        valid_count = safeGet(f'valid_{name}')
        exposed = safeGet(f'pop_exposed_{name}')
        results[f'pop_exposed_{name}'] = ee.Algorithms.If(valid_count.gt(0), exposed, None)

    # Multi-hazard topic counts: None if no hazard data anywhere in the country
    global_valid = safeGet('valid_global')
    for n in range(1, 9):
        val = safeGet(f'pop_topic_ge_{n}')
        results[f'pop_topic_ge_{n}'] = ee.Algorithms.If(global_valid.gt(0), val, None)

    # Topic exposures: None if no valid data for that topic's hazards in the country
    for topic in topic_exposure_list:
        topic_valid = safeGet(f'valid_{topic}_topic')
        val = safeGet(f'pop_exposed_{topic}_topic')
        results[f'pop_exposed_{topic}_topic'] = ee.Algorithms.If(topic_valid.gt(0), val, None)

    # MHI percentile exposures: None if country has no MHI coverage
    mhi_valid_count = safeGet('valid_mhi')
    for p in mhi_percentiles:
        val = safeGet(f'pop_mhi_p{p}')
        results[f'pop_mhi_p{p}'] = ee.Algorithms.If(mhi_valid_count.gt(0), val, None)

    return ee.Feature(None, results)

finalResults = countryStats.map(process_feature)

In [8]:
# =========================================================
# 6. EXPORT RESULTS
# =========================================================

ee.batch.Export.table.toDrive(
  collection=finalResults,
  description='Hazard_Population_Exposure',
  folder='GEE_Exports',
  fileFormat='CSV'
).start()